In [ ]:
from pathlib import Path
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
CODEBASE_DIR = Path('/content/drive/MyDrive/predicting-the-future-effect-of-optimizer-memory')  
assert (CODEBASE_DIR / 'requirements.txt').is_file(), 'Upload and unzip the code folder to Drive first.'
os.chdir(CODEBASE_DIR)
sys.path.insert(0, str(CODEBASE_DIR))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])


In [ ]:
from rttp_grid.surrogate_baselines import SurrogateBaselineConfig

ROOT_NAME = 'results/rttp_cifar_surrogate_baselines'
RUN_NAME = 'rttp_cifar_surrogate_baselines'

ARCHITECTURE = 'smallcnn'  
SEEDS = (0,)               
HORIZON = 40
LEARNING_RATE = 0.03
MOMENTUM = 0.9
RADIUS_SCALE = 1e-3

RESPONSE_RANK = 16
CANDIDATE_COUNT = 32
PERIODIC_REFRESH_PERIODS = (2, 5, 10)
LOW_RANK_RESPONSE_RANKS = (2, 4, 8)
TRUNCATED_TAIL_LENGTHS = (5, 10, 20)

FORCE_BOUNDARY = False

cfg = SurrogateBaselineConfig(
    run_name=RUN_NAME,
    root_name=ROOT_NAME,
    architecture=ARCHITECTURE,
    seeds=SEEDS,
    horizon=HORIZON,
    learning_rate=LEARNING_RATE,
    momentum=MOMENTUM,
    radius_scale=RADIUS_SCALE,
    response_rank=RESPONSE_RANK,
    candidate_count=CANDIDATE_COUNT,
    periodic_refresh_periods=PERIODIC_REFRESH_PERIODS,
    low_rank_response_ranks=LOW_RANK_RESPONSE_RANKS,
    truncated_tail_lengths=TRUNCATED_TAIL_LENGTHS,
    force_boundary=FORCE_BOUNDARY,
)
cfg

In [ ]:
from rttp_grid.surrogate_baselines import run_surrogate_baseline_experiment

output_root = run_surrogate_baseline_experiment(cfg)
print('output_root:', output_root)

In [ ]:
import pandas as pd
from pathlib import Path

from rttp_grid.utils import get_root
root = get_root(cfg)
summary = pd.read_csv(root / 'tables' / 'surrogate_baseline_summary.csv')
tradeoff = pd.read_csv(root / 'tables' / 'surrogate_cost_quality_tradeoff.csv')
raw = pd.read_csv(root / 'tables' / 'surrogate_baseline_raw.csv')

display(summary)
display(tradeoff.sort_values(['endpoint_error_mean', 'total_seconds_mean']))
display(raw.head())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig_dir = root / 'figures'
fig_dir.mkdir(exist_ok=True)

plot = tradeoff.copy()
plot['label'] = plot['surrogate'].astype(str)
for col in ['period', 'low_rank', 'tail_length']:
    if col in plot.columns:
        mask = plot[col].notna()
        plot.loc[mask, 'label'] += ':' + col + '=' + plot.loc[mask, col].astype(int).astype(str)

plt.figure(figsize=(9, 5))
plt.scatter(plot['total_seconds_mean'], plot['endpoint_error_mean'], s=70)
for _, row in plot.iterrows():
    plt.annotate(row['label'], (row['total_seconds_mean'], row['endpoint_error_mean']), fontsize=8, xytext=(4, 4), textcoords='offset points')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('total wall-clock seconds, log scale')
plt.ylabel('mean endpoint error, log scale')
plt.title('Surrogate cost-quality tradeoff')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(fig_dir / 'surrogate_cost_quality_tradeoff.png', dpi=220)
plt.savefig(fig_dir / 'surrogate_cost_quality_tradeoff.pdf')
plt.show()

plt.figure(figsize=(9, 5))
ordered = plot.sort_values('endpoint_error_mean')
plt.barh(ordered['label'], ordered['endpoint_error_mean'])
plt.xscale('log')
plt.xlabel('mean endpoint error, log scale')
plt.title('Surrogate prediction error')
plt.tight_layout()
plt.savefig(fig_dir / 'surrogate_endpoint_error.png', dpi=220)
plt.savefig(fig_dir / 'surrogate_endpoint_error.pdf')
plt.show()

In [ ]:
import shutil

archive = shutil.make_archive(str(root), 'zip', root)
print('archive:', archive)